# SPRINT 1

In [1]:
import pandas as pd

# Ler o arquivo CSV
df = pd.read_csv("halloween_vendas.csv", sep=",", encoding="utf-8")

# Conferir total de linhas
total_linhas = len(df)
print(f"O arquivo foi lido com sucesso! Total de linhas encontradas: {total_linhas}")

# Mostrar 3 registros
df.sample(3)


O arquivo foi lido com sucesso! Total de linhas encontradas: 1500


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
1060,310,2024-09-30 19:20:05,Chapeu de bruxa,Acessorio,Web,Maringá,3,22.0,8.00,0,Concluído,Todos,Unico,2024-10-02 19:20:05
830,531,2024-10-14 21:57:23,Fantasia de bruxa,Fantasia,Web,Maringá,2,110.0,57.00,0,Cancelado,Adulto,M,NaN
1015,777,2025-11-30 09:12:44,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,20,Concluído,Adulto,G,2025-12-04 09:12:44


# SPRINT 2 - Limpar e padronizar

In [2]:
import re

#  Vazio ou nulo (NaN) vira "sem categoria"
def limpar_categoria(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return "sem categoria"

    # Transforma dado em texto
    texto_str = str(texto)
        
    # Remove espaços em branco
    texto_sem_espacos = texto_str.strip()
        
    # Transforma todas as letras em minúsculas
    texto_minusculo = texto_sem_espacos.lower()
        
    # Tira pontuações
    texto_limpo = re.sub(r'[^a-z\s]', '', texto_minusculo)

    if texto_limpo.strip() == "":
        return "sem categoria"
        
    return texto_limpo

def padronizar_canal(texto):
    if pd.isna(texto):
        return "Não reconhecido"

    # Transforma dado em texto
    canal_str = str(texto)
    
    # Remove espaços em branco
    canal_sem_espacos = canal_str.strip()
    
    # Transforma todas as letras em minúsculas
    canal_minusculo = canal_sem_espacos.lower()

    #Padronizando tudo
       # #Padronizando tudo
    if "mercado" in canal_minusculo:
        return "Mercado"
    elif "loja" in canal_minusculo or "fisica" in canal_minusculo or "física" in canal_minusculo:
        return "Loja física"
    elif "web" in canal_minusculo:
        return "Web"
    else:
        return "Não reconhecido"

# Contagem:

# Cópia da coluna original para conseguir comparar o antes/depois
categorias_originais = df['categoria'].copy()

# Novas com os dados limpos para não estragar os originais
df['categoria_limpa'] = df['categoria'].apply(limpar_categoria)
df['canal_venda_limpo'] = df['canal_venda'].apply(padronizar_canal)

# Categorias vazias preenchidas
vazias_preenchidas = ((categorias_originais.isna()) | (categorias_originais.astype(str).str.strip() == "")).sum()

# Categorias não vazias alteradas
alteradas = 0
for orig, limpo in zip(categorias_originais, df['categoria_limpa']):
    if pd.notna(orig) and str(orig).strip() != "":
        if str(orig) != limpo:
            alteradas += 1

# Resultados:

print("--- RESULTADOS SPRINT 2 ---")
print(f"Categorias vazias preenchidas: {vazias_preenchidas}")
print(f"Categorias não vazias alteradas: {alteradas}")
print("\nExemplo de antes e depois dos dados na tabela:")
df[['categoria', 'categoria_limpa', 'canal_venda', 'canal_venda_limpo']].head(10)


--- RESULTADOS SPRINT 2 ---
Categorias vazias preenchidas: 18
Categorias não vazias alteradas: 1482

Exemplo de antes e depois dos dados na tabela:


,categoria,categoria_limpa,canal_venda,canal_venda_limpo
0,Fantasia,fantasia,Loja física,Loja física
1,FANTASIA!!!,fantasia,Loja física,Loja física
2,Fantasia,fantasia,Web,Web
3,Acessorio,acessorio,Loja física,Loja física
4,Acessorio,acessorio,Mercado,Mercado
5,Fantasia,fantasia,Loja física,Loja física
6,Fantasia,fantasia,Web,Web
7,Acessorio,acessorio,Loja física,Loja física
8,Fantasia,fantasia,Web,Web
9,Fantasia,fantasia,Loja física,Loja física


# SPRINT 3

In [3]:
#Identificar problemas na coluna Quantidade (vazia/NaN ou menor/igual a zero)
erros_quantidade = (df['quantidade'].isna()) | (df['quantidade'] <= 0)

#Identificar os problemas na coluna Preço Unitário (vazia/NaN ou menor que zero)
erros_preco = (df['preco_unitario'].isna()) | (df['preco_unitario'] < 0)

#Problemas individuais de cada campo 
total_erros_qtd = erros_quantidade.sum()
total_erros_preco = erros_preco.sum()

#Total de pedidos afetados sem sobreposição
pedidos_afetados_total = (erros_quantidade | erros_preco).sum()

#Nova coluna para marcar se o registro tem algum problema numérico
df['com_problema_numerico'] = erros_quantidade | erros_preco

print("--- RESULTADOS SPRINT 3 ---")
print(f"Pedidos com problemas na Quantidade: {total_erros_qtd}")
print(f"Pedidos com problemas no Preço Unitário: {total_erros_preco}")
print(f"Total de pedidos afetados (sem sobreposição): {pedidos_afetados_total}")

--- RESULTADOS SPRINT 3 ---
Pedidos com problemas na Quantidade: 7
Pedidos com problemas no Preço Unitário: 12
Total de pedidos afetados (sem sobreposição): 19
